In [2]:
import pandas as pd

In [3]:
def rename_columns(df, old_cols, new_cols):
    """
    Rename a series of columns using two aligned lists.
    """
    if len(old_cols) != len(new_cols):
        raise ValueError("old_cols and new_cols must have the same length")

    return df.rename(columns=dict(zip(old_cols, new_cols)))

### Transformer

In [59]:
# if args.type == 'txfr':

        ### transmission asset

file_loc = r"TXFR-T_REPLACEMENT LIST_01_12_26.xlsm"
df = pd.read_excel(file_loc, sheet_name = "Mar 2026 Repl Ranking by Phase", index_col=None, na_values=['NA'], skiprows=0)

sel_cols = ['SAP Equipment ID', 'Sub Name', 'Single or Three Phase', 'Highest Cumulative POF ',
            'Reliability Risk Matrix (PoF,CoF)']

df_t_txfr = df[sel_cols]

df_t_txfr=rename_columns(df_t_txfr, ['SAP Equipment ID', 'Sub Name','Single or Three Phase' , 'Highest Cumulative POF '], 
                                ['Equipment', 'Substation Name',  '1_3', 'POF'])

df_t_txfr['line'] = 'T'

    ### distribution asset

file_loc = r"D_Transf_Risk_Ranking_Under_Const 2026.xlsx"
df = pd.read_excel(file_loc, sheet_name = "Distribution Transformers", index_col=None, na_values=['NA'], skiprows=0, engine='openpyxl')

sel_cols = ['SAP Equipment I.D.', 'Substation Name', 'Single or Three Phase', 'Highest POF ','Reliability Risk Matrix (PoF,CoF)']

df_d_txfr = df[sel_cols]

df_d_txfr=rename_columns(df_d_txfr, ['SAP Equipment I.D.', 'Single or Three Phase', 'Highest POF '], 
                        ['Equipment', '1_3', 'POF'])

df_d_txfr['line'] = 'D'
# if args.line ==1:
asset_files = pd.concat([df_d_txfr, df_t_txfr])

asset_files = asset_files[~((asset_files['Equipment'].isin([41158251,
41158253,
41158254,
44918680,
45247388,
45274594])) & (asset_files['line']=='T'))]

asset_files = asset_files.dropna(subset=['Equipment'])
asset_files['Equipment'] = asset_files['Equipment'].astype(int)

In [24]:
asset_files[(asset_files['1_3']==1) & (asset_files['line']=='T')]

,Equipment,Substation Name,1_3,POF,"Reliability Risk Matrix (PoF,CoF)",line
19,42735593,ARCO SUB,1,0.014222,14,T
20,42735586,ARCO SUB,1,0.014548,14,T
21,43144171,ARCO SUB,1,0.014478,14,T
22,42735592,ARCO SUB,1,0.050000,24,T
23,43144172,ARCO SUB,1,0.014541,14,T
...,...,...,...,...,...,...
448,41014313,WHEELER RIDGE SUB,1,0.100000,33,T
451,41014343,WILSON SUB,1,0.040000,23,T
452,41012122,WILSON SUB,1,0.040000,23,T
453,41012121,WILSON SUB,1,0.040000,23,T


In [ ]:
.to_csv('transformer_discrepancy.csv', index=False)

In [101]:
pd.merge(pd.concat([df_d_txfr, df_t_txfr])[pd.concat([df_d_txfr, df_t_txfr])['Equipment'].isin([41158251,
41158253,
41158254,
44918680,
45247388,
45274594])].sort_values('Equipment'), df_txfr_foundary, right_on='sap_equipment_id' , left_on='Equipment', how='inner').to_csv('transformer_discrepancy.csv', index=False)

In [9]:
df = pd.read_excel(r"C:\Users\SIH5\OneDrive - PGE\Desktop\transformer failures\txfr_brkr_failures\2024 + Emergency Job Tracker_5_7_25.xlsm",
 sheet_name = "Emerg Tracker (TCR) 2024 +", index_col=None, na_values=['NA'], skiprows=3, engine='openpyxl')
sel_cols = ['SAP Equipment Number(s), if available', 'Project Name for SAP (Failed Asset)\n(max. 40 char.)','Equipment Type\n(being replaced)',
            'Voltage (kV)', 'Location\n(Station Name)','Future\nYear\nImpact\n($000)', 
            'Estimated Spend in Current Year\n($000)','Date SAM first contacted',
            'Date Declared Emergency (Date decision to be placed in emergency)',
           'Failure Type', 'In-Serv./Catastrophic Equipment Category (CB, Xfmr, Unitsub, 3ph Reg, Battery)',
           'Outage Customer Minutes (CMIN) In-Service and Catastrophic Failures (XFMR, CB, XFMR Bushings, Minor)',
           'Age at Year of Failure', 'MVA/KVA (Transformer & Regs)', '3Ph or 1Ph? (Transformers and Regs)']

df_fail = df[sel_cols]

df_fail = rename_columns(df_fail, sel_cols, ["sap_id", "Project Name", "Equipment Type", "Voltage", "Substation Name", 
                                            "Future Spending", "Current Spending","Discover Date", "Date", "Failure Type",
                                            "Catastrophic Equipment", "CMIN", "Age", "MVA/KVA", "3Ph or 1Ph"])

df_fail['Date'] = pd.to_datetime(df_fail['Discover Date'], errors='coerce')
df_fail['Year'] = df_fail['Date'].dt.year
df_fail['Month'] = df_fail['Date'].dt.month

c:\Users\SIH5\anaconda3\envs\sih5\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [10]:
df_fail = df_fail[~df_fail["Year"].isna()]

In [14]:
df_txfr =df_fail[(df_fail["Equipment Type"]=="TXFR") & (df_fail['Date']>=f'01-01-{str(2025)}') & (df_fail['Date']<f'01-01-{str(2026)}')]
t_count = df_txfr.shape[0]


df_txfr = df_txfr[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_txfr = rename_columns(df_txfr, ['sap_id'], ['Equipment'])

clean_df = df_txfr.dropna(subset=['Equipment'])
clean_df['Equipment'] = clean_df['Equipment'].astype(int)

In [15]:
clean_df

,Equipment,Equipment Type,Voltage,Substation Name,Future Spending,Current Spending,Date,Age,Failure Type
275,41180092,TXFR,12,PHILO SUB,NaN,3000,2025-01-10,22,JIT
276,41013672,TXFR,12,PHILO SUB,NaN,3000,2025-01-10,73,JIT
277,41013678,TXFR,12,PHILO SUB,NaN,3000,2025-01-10,73,JIT
279,41013680,TXFR,12,PHILO SUB,NaN,3000,2025-01-10,NaN,JIT
280,41014614,TXFR,60/12,SHADY GLEN SUB,2000,150,2025-01-22,74,JIT
...,...,...,...,...,...,...,...,...,...
718,41013073,TXFR,60/12,Grass Valley,500,1000,2025-10-28,NaN,JIT
719,41013381,TXFR,60/12,Marysville,500,1000,2025-10-28,NaN,JIT
720,41013379,TXFR,60/12,Marysville,500,1000,2025-10-28,NaN,JIT
721,41013378,TXFR,60/12,Marysville,500,1000,2025-10-28,NaN,JIT


In [87]:
not_in_txfr = pd.merge(clean_df, asset_files, on='Equipment', how='left')[pd.merge(clean_df, asset_files, on='Equipment', how='left')['line'].isna()]

In [88]:
in_txfr = pd.merge(clean_df, asset_files, on='Equipment', how='inner')

In [82]:
len(in_txfr[in_txfr['line']=='D'])

8

In [89]:
not_in_txfr.to_csv('failed_not_in_replacement_file.csv', index=False)

In [69]:
doc = '('
for col in range(len(not_in_txfr)):
    doc+=f"'{str(not_in_txfr.iloc[col, 0])}',"

In [70]:
doc

"('41013680','41014613','41014510','41014952','41014465','41013148','41013196','41014468','41013077','41014299','41014301','41195082','40989444','41014782','41014228','41014225','41012942','41340841','41012397','41015245','5535264','41012205',"

In [84]:
asset_files[asset_files['Equipment'].isin([41013680,
41014613,41014510,41014952,41014465,41013148,41013196,41014468,41013077,41014299,41014301,41195082,40989444,41014782,41014228,41014225,41012942,41340841,41012397,41015245,5535264,
41012205]
)]

,Equipment,Substation Name,1_3,POF,"Reliability Risk Matrix (PoF,CoF)",line


In [90]:
df_txfr_foundary = pd.read_csv('transformers_foundary.csv')

In [91]:
df_txfr_foundary['sap_equipment_id'] = df_txfr_foundary['sap_equipment_id'].astype(int)

In [55]:
not_in_txfr = not_in_txfr.dropna(subset =['Equipment'])

In [92]:
not_in_txfr['Equipment'] = not_in_txfr['Equipment'].astype(int)

In [105]:
pd.merge(not_in_txfr, df_txfr_foundary, right_on='sap_equipment_id', left_on='Equipment', how='left').to_csv('failed_not_in_replacement_file.csv', index=False)

In [49]:
not_in_txfr

,Equipment,Equipment Type,Voltage,Substation Name_x,Future Spending,Current Spending,Date,Age,Failure Type,Substation Name_y,1_3,POF,"Reliability Risk Matrix (PoF,CoF)",line
3,41013680,TXFR,12,PHILO SUB,NaN,3000,2025-01-10,NaN,JIT,NaN,NaN,NaN,NaN,NaN
4,41014614,TXFR,60/12,SHADY GLEN SUB,2000,150,2025-01-22,74,JIT,NaN,NaN,NaN,NaN,NaN
5,41014613,TXFR,60/12,SHADY GLEN SUB,2000,150,2025-01-22,74,JIT,NaN,NaN,NaN,NaN,NaN
6,41014615,TXFR,60/12,SHADY GLEN SUB,1000,150,2025-01-22,74,JIT,NaN,NaN,NaN,NaN,NaN
10,41013579,TXFR,60/12,PACIFICA SUB,5000,500,2025-01-02,55,JIT,NaN,NaN,NaN,NaN,NaN
13,\n41012312,TXFR,60,INDIAN FLAT SUB,2500,500,2025-01-24,75,JIT,NaN,NaN,NaN,NaN,NaN
14,41014510,TXFR,115/12,MILLBRAE SUB,50,7000,2025-02-14,40,JIT,NaN,NaN,NaN,NaN,NaN
21,41014952,TXFR,21/12,BAKERSFIELD SUB,NaN,1000,2025-01-31,55,JIT,NaN,NaN,NaN,NaN,NaN
22,41014465,TXFR,70/12,SAN BERNARD SUB,NaN,1000,2025-03-27,73,JIT,NaN,NaN,NaN,NaN,NaN
24,41012053,TXFR,230,CARIBOU PH #2,5000,750,2025-01-24,0,JIT,NaN,NaN,NaN,NaN,NaN


In [108]:
txfr_not_in_foundary = pd.merge(asset_files, df_txfr_foundary, left_on ='Equipment' , right_on='sap_equipment_id', how='left')[
    pd.merge(asset_files, df_txfr_foundary, left_on ='Equipment' , right_on='sap_equipment_id', how='left')['sap_equipment_id'].isna()]

In [109]:
doc = '('
for col in range(len(txfr_not_in_foundary)):
    doc+=f"'{str(txfr_not_in_foundary.iloc[col, 0])}',"

In [110]:
doc

"('42735572','43144204','42553779','42734697','41184033','40989612','42686386','42686875','40989631','42692025','40989656','42626324','44611881','40989714','42695718','40989681','42710844','40989524','40989800',"

In [111]:
txfr_not_in_foundary.to_csv('txfr_not_in_foundary.csv', index=False)

### CB

In [4]:
### transmission asset
file_loc = r"T_BRKR 1-N Under Const (1 15 2026).xlsx"
df = pd.read_excel(file_loc, sheet_name = "Transmission Breakers", index_col=None, na_values=['NA'], skiprows=0)

sel_cols = [' Equipment', 'Substation Name', 'Highest PoF',
            'Reliability Risk Matrix (PoF,CoF)']

df_t_brkr = df[sel_cols]


df_t_brkr=rename_columns(df_t_brkr, [' Equipment', 'Highest PoF'], 
                        ['Equipment', 'POF'])

df_t_brkr['line'] = 'T'
### distribution asset
file_loc = r"D BRKR 1-N Under Const 2026.xlsx"
df = pd.read_excel(file_loc, sheet_name = "Distribution Breakers", index_col=None, na_values=['NA'], skiprows=0, engine='openpyxl')

sel_cols = ['Equipment', 'Substation Name', 'Highest Cummulative PoF',
            'Reliability Risk Matrix (PoF,CoF)']

df_d_brkr = df[sel_cols]

df_d_brkr=rename_columns(df_d_brkr, ['Highest Cummulative PoF'], 
                        ['POF'])


df_d_brkr['line'] = 'D'


asset_files = pd.concat([df_d_brkr, df_t_brkr])
asset_files = asset_files[~((asset_files['Equipment'].isin([40991421])) & (asset_files['line']=='T'))]

asset_files = asset_files.dropna(subset=['Equipment'])
asset_files['Equipment'] = asset_files['Equipment'].astype(int)

In [36]:
df_cb_foundary = pd.read_csv('circuitbreaker_foundary.csv')
df_cb_foundary = df_cb_foundary.dropna(subset='sap_equipment_id')
df_cb_foundary['sap_equipment_id'] = df_cb_foundary['sap_equipment_id'].astype(int)

C:\Users\SIH5\AppData\Local\Temp\ipykernel_15196\4271463748.py:1: DtypeWarning: Columns (4,49,57,60,72,79,80,92,117,122,139,144,163,164,167,168,188,189,204,212,213,222,232,233,248,249,256,257) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cb_foundary = pd.read_csv('circuitbreaker_foundary.csv')


In [7]:
pd.merge(pd.concat([df_d_brkr, df_t_brkr])[pd.concat([df_d_brkr, df_t_brkr])['Equipment'].isin([40991421])].sort_values('Equipment'),
 df_cb_foundary, right_on='sap_equipment_id' , left_on='Equipment', how='inner').to_csv('cb_discrepancy.csv', index=False)

In [40]:
df_txfr =df_fail[(df_fail["Equipment Type"]=="BRKR") & (df_fail['Date']>=f'01-01-{str(2026)}') & (df_fail['Date']<f'01-01-{str(2027)}')]
t_count = df_txfr.shape[0]


df_txfr = df_txfr[["sap_id", "Equipment Type", "Voltage", "Substation Name", 
                                        "Future Spending", "Current Spending", "Date", "Age", "Failure Type"]]


df_txfr = rename_columns(df_txfr, ['sap_id'], ['Equipment'])

clean_df = df_txfr.dropna(subset=['Equipment'])
clean_df['Equipment'] = clean_df['Equipment'].astype(int)

C:\Users\SIH5\AppData\Local\Temp\ipykernel_15196\2625886312.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df['Equipment'] = clean_df['Equipment'].astype(int)


In [41]:
len(clean_df)

19

In [42]:
not_in_brkr = pd.merge(clean_df, asset_files, on='Equipment', how='left')[pd.merge(clean_df, asset_files, on='Equipment', how='left')['line'].isna()]

In [43]:
in_brkr = pd.merge(clean_df, asset_files, on='Equipment', how='inner')

In [29]:
len(in_brkr[in_brkr['line']=='D'])

5

In [34]:
not_in_brkr.to_csv('failed_not_in_replacement_file_2025.csv', index=False)

In [44]:
pd.merge(not_in_brkr, df_cb_foundary, right_on='sap_equipment_id', left_on='Equipment', how='left').to_csv('failed_not_in_replacement_file_brkr_2026.csv', index=False)

In [45]:
brkr_not_in_foundary = pd.merge(asset_files, df_cb_foundary, left_on ='Equipment' , right_on='sap_equipment_id', how='left')[
    pd.merge(asset_files, df_cb_foundary, left_on ='Equipment' , right_on='sap_equipment_id', how='left')['sap_equipment_id'].isna()]

In [47]:
brkr_not_in_foundary.to_csv('brkr_not_in_foundary.csv', index=False)